In [3]:
import os
import json
import time
import random
from typing import Optional, Any
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda

In [4]:
# 加载环境变量
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError(
        "\n请先在 .env 文件中设置有效的 GROQ_API_KEY\n"
        "访问 https://console.groq.com/keys 获取免费密钥"
    )
model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

# JSON 解析辅助函数

In [6]:
def safe_parse_json(text: str, default: dict = None) -> dict:
    """
    安全地解析JSON文本

    处理：
    - Markdown 代码块 (```json ... ```)
    - 前后的空白字符
    - 解析失败时返回默认值
    """
    if default is None:
        default = {}

    content = text.strip()

    # 移除 Markdown 代码块
    if "```json" in content:
        try:
            content = content.split("```json")[1].split("```")[0]
        except IndexError:
            pass
    elif "```" in content:
        try:
            parts = content.split("```")
            if len(parts) >= 2:
                content = parts[1]
        except IndexError:
            pass

    content = content.strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        print(f"   ⚠️ JSON 解析失败: {e}")
        return default

# 基本重试机制

In [7]:
def basic_retry():
    """
    实现基本的重试机制
    """
    print("\n"+"="*60)
    print("基本重试机制")
    print("="*60)

    def retry_with_backoff(
            func,
            max_retries: int = 3,
            base_delay: float = 1.0,
            max_delay: float = 60.0
    ):
        """
        带指数退避的重试装饰器
        """
        def wrapper(*args, **kwargs):
            last_exception = None

            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e

                    if attempt < max_retries - 1:
                        # 计算等待时间（指数退避 + 随机抖动）
                        delay = min(base_delay * (2 ** attempt), max_delay)
                        jitter = random.uniform(0, delay * 0.1)
                        wait_time = delay + jitter

                        print(f"  ⚠️ 尝试 {attempt + 1} 失败: {e}")
                        print(f"     等待 {wait_time:.1f} 秒后重试...")
                        time.sleep(wait_time)
                    else:
                        print(f"  ❌ 所有 {max_retries} 次尝试均失败")

            raise last_exception

        return wrapper

    # 模拟不稳定的函数
    call_count = [0]

    def unstable_function(query: str):
        """模拟一个可能失败的函数"""
        call_count[0] +=1
        # 前两次调用失败
        if call_count[0] <= 2:
            raise ConnectionError(f"模拟网络错误(尝试{call_count[0]})")
        return model.invoke(query)

    # 应用重试
    stable_function = retry_with_backoff(unstable_function, max_retries=3, base_delay=0.5)
    print("调用不稳定函数（前2次会失败）...")
    try:
        result = stable_function("简单回答：1+1等于几？")
        print(f"  ✅ 最终成功: {result.content}")
    except Exception as e:
        print(f"  ❌ 最终失败: {e}")

# 模型回退机制

In [14]:
def model_fallback():
    """
    实现模型回退：主模型失败时使用备用模型
    """
    print("\n" + "=" * 60)
    print("模型回退机制")
    print("=" * 60)

    class FallbackChain:
        """带回退的模型链"""
        def __init__(self, models: list):
            self.models = models
        def invoke(self, query: str) -> Any:
            last_error = None
            for i, model in enumerate(self.models):
                model_name = f"模型 {i + 1}"
                try:
                    print(f"尝试{model_name}")
                    result = model.invoke(query)
                    print(f"{model_name}成功")
                    return result
                except Exception as e:
                    last_error = e
                    print(f"{model_name}失败: {e}")
            raise Exception(f"所有模型都失败：{last_error}")

    models = [
        model,
        model,
        model
    ]
    falllback_Chain = FallbackChain(models)
    print("使用回退链调用...")
    result = falllback_Chain.invoke("简单回答：1+1等于几？")
    print(f"结果：{result.content}")


# 输出验证和修复

In [10]:
def output_validation():
    """
    验证 LLM 输出并在需要时重试和修复
    """
    print("\n" + "=" * 60)
    print("输出验证和修复")
    print("=" * 60)
    class ProductInfo(BaseModel):
        """产品信息结构"""
        name: str = Field(description="产品名称")
        price: float = Field(gt=0, description="产品价格(必须大于0)")
        category: str = Field(description="产品类别")

    def extract_product_info(description: str, max_retries: int = 3) -> ProductInfo:
        """
        从描述中提取产品信息，带验证和重试
        """
        prompt = f"""从以下描述中提取产品信息，返回 JSON 格式：
{{"name": "产品名称", "price": 数字价格, "category": "类别"}}

描述: {description}

只返回 JSON，不要其他内容。"""
        for attempt in range(max_retries):
            try:
                response = model.invoke(prompt)
                content = response.content.strip()
                # 使用安全的 JSON 解析
                data = safe_parse_json(content, None)
                if data in None:
                    raise json.JSONDecodeError("解析失败", content, 0)
                # 验证数据
                product = ProductInfo(**data)
                print(f"  ✅ 验证通过 (尝试 {attempt + 1})")
                return product
            except json.JSONDecodeError as e:
                print(f"  ⚠️ JSON 解析失败: {e}")
                print(f"   尝试 {attempt + 1} 失败，将重试...")
            except ValidationError as e:
                print(f"  ⚠️ 验证失败: {e}")
                print(f"   尝试 {attempt + 1} 失败，将重试...")
            except Exception as e:
                print(f"  ⚠️ 验证失败: {e}")
                print(f"   尝试 {attempt + 1} 失败，将重试...")

        # 返回默认值
        print("  ❌ 验证失败，返回默认值")
        return ProductInfo(name="未知产品", price=0.01, category="未分类")

    # 测试
    description = [
       "我们的新款蓝牙耳机售价299元，属于电子产品类别",
        "这是一个测试"  # 可能导致解析问题
    ]
    for desc in description:
        print(f"\n描述：{desc}")
        product = extract_product_info(desc)
        print(f"  结果: {product}")

# 超时处理

In [11]:
def timeout_handling():
    """
    实现请求超时处理
    """
    print("\n" + "=" * 60)
    print("超时处理")
    print("=" * 60)

    import concurrent.futures

    def invoke_with_timeout(model, query: str, timeout: float = 30.0):
        """
        带超时的模型调用
        """
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(model.invoke, query)
            try:
                result = future.result(timeout=timeout)
                return {"success": True, "content": result.content}
            except concurrent.futures.TimeoutError:
                return {
                    "success": False,
                    "error": "请求超时",
                    "timeout": timeout
                }

    # 测试
    print("测试带超时的调用（超时设置：30秒）...")
    result = invoke_with_timeout(model, "你好！", timeout=30.0)
    print(f"  结果: {result}")

In [12]:
def main():
    print("\n"+"="*70)
    print("LangChain 错误处理")
    print("="*70)

    basic_retry()
    model_fallback()
    output_validation()
    timeout_handling()

In [15]:
if __name__ == "__main__":
    main()


LangChain 错误处理

基本重试机制
调用不稳定函数（前2次会失败）...
  ⚠️ 尝试 1 失败: 模拟网络错误(尝试1)
     等待 0.5 秒后重试...
  ⚠️ 尝试 2 失败: 模拟网络错误(尝试2)
     等待 1.1 秒后重试...
  ✅ 最终成功: 2

模型回退机制
使用回退链调用...
尝试模型 1
模型 1成功
结果：2

输出验证和修复

描述：我们的新款蓝牙耳机售价299元，属于电子产品类别
  ⚠️ 验证失败: argument of type 'NoneType' is not iterable
   尝试 1 失败，将重试...
  ⚠️ 验证失败: argument of type 'NoneType' is not iterable
   尝试 2 失败，将重试...
  ⚠️ 验证失败: argument of type 'NoneType' is not iterable
   尝试 3 失败，将重试...
  ❌ 验证失败，返回默认值
  结果: name='未知产品' price=0.01 category='未分类'

描述：这是一个测试
  ⚠️ 验证失败: argument of type 'NoneType' is not iterable
   尝试 1 失败，将重试...
  ⚠️ 验证失败: argument of type 'NoneType' is not iterable
   尝试 2 失败，将重试...
  ⚠️ 验证失败: argument of type 'NoneType' is not iterable
   尝试 3 失败，将重试...
  ❌ 验证失败，返回默认值
  结果: name='未知产品' price=0.01 category='未分类'

超时处理
测试带超时的调用（超时设置：30秒）...
  结果: {'success': True, 'content': '你好。今天我能为您做什么？'}
